[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/04_control.ipynb)

# Tutorial 4: Actuators & Control

Learn how to add actuators, implement controllers, and use `data.ctrl`.

In [ ]:
!pip install -q mujoco mediapy matplotlib

In [ ]:
import mujoco
import numpy as np
import matplotlib.pyplot as plt
import mediapy as media

## Actuators in MJCF

Actuators apply forces/torques to joints. Common types:
- `<motor>` — direct torque
- `<position>` — position servo
- `<velocity>` — velocity servo

In [ ]:
xml = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 2"/>
    <body pos="0 0 1.5">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.04" mass="1" rgba="0 0.7 0 1"/>
    </body>
  </worldbody>
  <actuator>
    <motor name="torque" joint="hinge" ctrlrange="-5 5"/>
  </actuator>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)
print(f"Actuators: {model.nu}, Joints: {model.njnt}")

## PD Controller

$$\tau = k_p (q_{target} - q) - k_d \dot{q}$$

In [ ]:
target_angle = np.pi / 3  # 60 degrees
kp, kd = 10.0, 2.0

times, angles, controls = [], [], []
mujoco.mj_resetData(model, data)

while data.time < 5.0:
    # PD control
    angle = data.qpos[0]
    velocity = data.qvel[0]
    data.ctrl[0] = np.clip(kp * (target_angle - angle) - kd * velocity, -5, 5)
    
    mujoco.mj_step(model, data)
    times.append(data.time)
    angles.append(angle)
    controls.append(data.ctrl[0])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(times, np.degrees(angles), label='Angle')
ax1.axhline(np.degrees(target_angle), color='r', linestyle='--', label='Target')
ax1.set_ylabel('Angle (deg)'); ax1.legend(); ax1.grid(True)
ax2.plot(times, controls, color='orange')
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Torque (Nm)'); ax2.grid(True)
plt.suptitle('PD Control of Pendulum'); plt.tight_layout(); plt.show()

## Tuning PD Gains

Compare different gain settings:

In [ ]:
plt.figure(figsize=(10, 4))

for kp, kd, label in [(5, 1, 'Underdamped'), (10, 6, 'Critically damped'), (10, 15, 'Overdamped')]:
    mujoco.mj_resetData(model, data)
    times, angles = [], []
    while data.time < 5.0:
        data.ctrl[0] = np.clip(kp * (target_angle - data.qpos[0]) - kd * data.qvel[0], -5, 5)
        mujoco.mj_step(model, data)
        times.append(data.time)
        angles.append(data.qpos[0])
    plt.plot(times, np.degrees(angles), label=label)

plt.axhline(np.degrees(target_angle), color='k', linestyle='--', alpha=0.5)
plt.xlabel('Time (s)'); plt.ylabel('Angle (deg)')
plt.title('Effect of PD Gains'); plt.legend(); plt.grid(True); plt.show()

## Position & Velocity Servos

MuJoCo has built-in servo actuators that implement PD internally:

In [ ]:
xml_servo = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 2"/>
    <body pos="0 0 1.5">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.04" mass="1" rgba="0.8 0.3 0.1 1"/>
    </body>
  </worldbody>
  <actuator>
    <!-- Position servo: ctrl = desired position, kp is gain -->
    <position name="pos_servo" joint="hinge" kp="10" ctrlrange="-3.14 3.14"/>
  </actuator>
</mujoco>
"""

model_s = mujoco.MjModel.from_xml_string(xml_servo)
data_s = mujoco.MjData(model_s)

# Just set the target — the servo does the rest
times, angles = [], []
while data_s.time < 5.0:
    # Step target over time
    data_s.ctrl[0] = np.pi/4 if data_s.time < 2.5 else -np.pi/4
    mujoco.mj_step(model_s, data_s)
    times.append(data_s.time)
    angles.append(data_s.qpos[0])

plt.figure(figsize=(10, 4))
plt.plot(times, np.degrees(angles))
plt.xlabel('Time (s)'); plt.ylabel('Angle (deg)')
plt.title('Position Servo (built-in PD)'); plt.grid(True); plt.show()

## Velocity Servo

A velocity servo applies force proportional to velocity error:
$$F = k_v (v_{target} - v)$$

Set `ctrl` = desired velocity. Useful for wheels, conveyors, continuous rotation.

In [ ]:
xml_vel = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 2"/>
    <body pos="0 0 1.5">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.04" mass="1" rgba="0.1 0.5 0.8 1"/>
    </body>
  </worldbody>
  <actuator>
    <velocity name="vel_servo" joint="hinge" kv="5" ctrlrange="-5 5"/>
  </actuator>
</mujoco>
"""

model_v = mujoco.MjModel.from_xml_string(xml_vel)
data_v = mujoco.MjData(model_v)

# Command constant velocity
times, vels = [], []
while data_v.time < 5.0:
    data_v.ctrl[0] = 2.0  # desired angular velocity (rad/s)
    mujoco.mj_step(model_v, data_v)
    times.append(data_v.time)
    vels.append(data_v.qvel[0])

plt.figure(figsize=(10, 3))
plt.plot(times, vels)
plt.axhline(2.0, color='r', linestyle='--', label='Target velocity')
plt.xlabel('Time (s)'); plt.ylabel('Velocity (rad/s)')
plt.title('Velocity Servo'); plt.legend(); plt.grid(True); plt.show()

## Actuator Types Comparison

| Type | `ctrl` meaning | Internal formula | Use case |
|------|---------------|-----------------|----------|
| `<motor>` | Direct force/torque | $F = \text{gear} \cdot \text{ctrl}$ | Full control authority (RL, custom controllers) |
| `<position>` | Target position | $F = k_p (\text{ctrl} - q)$ | Servos, joint-space tracking |
| `<velocity>` | Target velocity | $F = k_v (\text{ctrl} - \dot{q})$ | Wheels, conveyors |
| `<cylinder>` | Piston pressure | Pneumatic dynamics | Hydraulic/pneumatic actuators |
| `<muscle>` | Activation (0-1) | Force-length-velocity curves | Biomechanical models |
| `<general>` | Configurable | Customizable transmission | Advanced custom actuators |

All actuators can have `ctrlrange`, `forcerange`, `gear`, and `dyntype` attributes.

## General Actuator (Custom Transmission)

The `<general>` actuator lets you configure gain, bias, and dynamics independently.
This is the most flexible actuator type — all others are special cases of it.

In [ ]:
xml_general = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 2"/>
    <body pos="0 0 1.5">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.04" mass="1" rgba="0.7 0.2 0.7 1"/>
    </body>
  </worldbody>
  <actuator>
    <!-- General actuator configured as position servo with damping -->
    <!-- gainprm[0]=kp, biasprm[1]=kp (negative feedback), biasprm[2]=kd -->
    <general name="custom" joint="hinge"
             gainprm="10" biasprm="0 -10 -2"
             ctrlrange="-3.14 3.14"/>
  </actuator>
</mujoco>
"""

model_g = mujoco.MjModel.from_xml_string(xml_general)
data_g = mujoco.MjData(model_g)

# This behaves like position servo: ctrl=target, internal PD
times, angles = [], []
while data_g.time < 5.0:
    data_g.ctrl[0] = np.pi / 3
    mujoco.mj_step(model_g, data_g)
    times.append(data_g.time)
    angles.append(data_g.qpos[0])

plt.figure(figsize=(10, 3))
plt.plot(times, np.degrees(angles))
plt.axhline(60, color='r', linestyle='--', label='Target')
plt.xlabel('Time (s)'); plt.ylabel('Angle (deg)')
plt.title('General Actuator (custom PD)'); plt.legend(); plt.grid(True); plt.show()

## Muscle Actuator

MuJoCo has a biomechanical muscle model with:
- **Activation dynamics** (first-order filter)
- **Force-length curve** (bell-shaped, peak at optimal length)
- **Force-velocity curve** (concentric weaker than eccentric)

Control input is excitation ∈ [0, 1]. The muscle filters this into activation.

In [ ]:
xml_muscle = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <default>
    <muscle ctrlrange="0 1"/>
  </default>
  <worldbody>
    <light pos="0 0 2"/>
    <body pos="0 0 1.5">
      <joint name="hinge" type="hinge" axis="0 1 0" range="-90 90"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.4" size="0.03" mass="1" rgba="0.9 0.6 0.5 1"/>
    </body>
  </worldbody>
  <tendon>
    <fixed name="muscle_tendon">
      <joint joint="hinge" coef="0.05"/>
    </fixed>
  </tendon>
  <actuator>
    <muscle name="flexor" tendon="muscle_tendon" timeconst="0.04 0.1"/>
  </actuator>
</mujoco>
"""

model_m = mujoco.MjModel.from_xml_string(xml_muscle)
data_m = mujoco.MjData(model_m)

# Pulse excitation
times, angles, activations = [], [], []
while data_m.time < 3.0:
    # Full excitation for 1s, then release
    data_m.ctrl[0] = 1.0 if data_m.time < 1.0 else 0.0
    mujoco.mj_step(model_m, data_m)
    times.append(data_m.time)
    angles.append(np.degrees(data_m.qpos[0]))
    activations.append(data_m.act[0])  # internal activation state

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax1.plot(times, angles)
ax1.set_ylabel('Angle (deg)'); ax1.set_title('Muscle Actuator'); ax1.grid(True)
ax2.plot(times, activations, color='red', label='Activation')
ax2.axvspan(0, 1, alpha=0.1, color='green', label='Excitation ON')
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Activation'); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()
print('Note: Activation lags behind excitation (first-order dynamics)')

## Keyframes

Keyframes store named configurations (qpos/qvel/ctrl) you can reset to.
Useful for setting initial poses, test configurations, or waypoints.

In [ ]:
xml_keyframe = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 2"/>
    <body pos="0 0 1.5">
      <joint name="j1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0.3 0 0" size="0.02" mass="1" rgba="0.3 0.6 0.9 1"/>
      <body pos="0.3 0 0">
        <joint name="j2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0.25 0 0" size="0.015" mass="0.7" rgba="0.4 0.8 0.3 1"/>
      </body>
    </body>
  </worldbody>
  <actuator>
    <position joint="j1" kp="20"/>
    <position joint="j2" kp="15"/>
  </actuator>
  <keyframe>
    <key name="home" qpos="0 0"/>
    <key name="reach_up" qpos="-0.5 0.8"/>
    <key name="reach_forward" qpos="0.3 -0.6"/>
  </keyframe>
</mujoco>
"""

model_k = mujoco.MjModel.from_xml_string(xml_keyframe)
data_k = mujoco.MjData(model_k)

# Reset to each keyframe and render
renderer = mujoco.Renderer(model_k, height=200, width=300)
images = []
for i in range(model_k.nkey):
    key_name = mujoco.mj_id2name(model_k, mujoco.mjtObj.mjOBJ_KEY, i)
    # Load keyframe
    mujoco.mj_resetDataKeyframe(model_k, data_k, i)
    mujoco.mj_forward(model_k, data_k)
    renderer.update_scene(data_k)
    images.append(renderer.render().copy())
    print(f'Keyframe "{key_name}": qpos = {data_k.qpos}')

renderer.close()
media.show_images(images, titles=['home', 'reach_up', 'reach_forward'])

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| `<motor>` | Direct torque — you provide the full control signal |
| `<position>` | Built-in PD servo — just set target angle |
| `<velocity>` | Velocity tracking — set target speed |
| `<general>` | Fully configurable gain/bias/dynamics |
| `<muscle>` | Biomechanical with activation dynamics |
| `data.ctrl` | Array of control inputs (one per actuator) |
| `ctrlrange` | Clips control input; `forcerange` clips output force |
| Keyframes | Named poses for reset (`mj_resetDataKeyframe`) |

**Next:** [Tutorial 05 — Contact & Collision](05_contact.ipynb)